# MDLM 实验 Notebook — 扩散过程可视化 & 采样器对比

**Spring 2026《随机过程》课程项目**

基于 MDLM (Masked Diffusion Language Model, Sahoo et al., NeurIPS 2024) 的推理实验。

---

## 包含实验

| # | 实验 | 内容 |
|:-:|------|------|
| 1a | **多 Seed 可视化** | 3 个随机种子下的去噪过程对比 (seed=42, 123, 999) |
| 1b | **步数对比** | 128 / 500 / 1000 步对生成质量的影响 |
| 2 | **采样器效率对比** | ddpm_cache / ddpm / analytic 三者的速度与 PPL |

---

## 提升生成质量的技巧

MDLM 的生成质量受以下因素影响：
1. **采样步数**：越多越好，推荐 1000 步
2. **随机种子**：不同 seed 差异很大，多试几个
3. **序列长度**：256 是平衡质量和速度的选择

> ⏱ T4 GPU 上 1000 步约需 3-5 分钟

---
## 0. 安装依赖 & 设置环境

In [ ]:
import os, sys, time, json, itertools, math, zipfile, io

# 安装依赖
# 这大概需要 3 分钟
# 忽略 pip 的依赖 error
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers==4.38.2 datasets einops==0.7.0 -q
!pip install lightning==2.2.1 hydra-core==1.3.2 omegaconf==2.3.0 gdown -q
!pip install fsspec h5py packaging==23.2 rich==13.7.1 -q
!pip install numpy==1.26.4 -q
!pip install triton -q

In [ ]:
# 安装 flash-attn（mdlm-owt 需要）
# 自动检测 Python 版本并选择正确的 wheel
import sys, subprocess
py_version = f"cp{sys.version_info.major}{sys.version_info.minor}"
print(f"Python version: {py_version}")

# flash-attn 2.7.3 + CUDA 12.x + torch 2.2 的预编译 wheel
FLASH_ATTN_URL = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/"
    f"flash_attn-2.7.3+cu12torch2.2cxx11abiFALSE-{py_version}-{py_version}-linux_x86_64.whl"
)

try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", FLASH_ATTN_URL, "-q"])
    print("flash-attn 安装成功 ✅")
except Exception as e:
    print(f"预编译 wheel 安装失败: {e}")
    print("尝试从源码编译安装...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"])


In [ ]:
import os
import zipfile
import io

os.chdir('/content')  # 强行切回 Colab 的根目录
EXTRACT_DIR = '/content/mdlm_chinese'
ZIP_FILE_PATH = '/content/mdlm_chinese.zip'

if os.path.exists(EXTRACT_DIR):
    print(f'代码已就绪: {EXTRACT_DIR}')
else:
  # 下载代码文件
  # 使用 gdown 下载指定 ID 的文件，并重命名为 mdlm_chinese.zip 放到当前 /content 目录下
  !gdown --id 1YsBEsTkvZ0eXWmD0odjVgppwTMN0cre9 -O {ZIP_FILE_PATH}

  # 确保解压目录存在
  os.makedirs(EXTRACT_DIR, exist_ok=True)

  # 解压 zip 文件
  with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zf:
      zf.extractall(EXTRACT_DIR)
  print(f'已解压 {ZIP_FILE_PATH} 到 {EXTRACT_DIR}')

  # 如果本地有 mdlm_chinese.zip 可以手动上传
  # from google.colab import files
  # print('请在弹出的对话框中选择 mdlm_chinese.zip ...')
  # uploaded = files.upload()  # 返回 {文件名: 字节内容}
  # for fname, content in uploaded.items():
  #     with zipfile.ZipFile(io.BytesIO(content)) as zf:
  #         zf.extractall(EXTRACT_DIR)
  #     print(f'已解压 {fname} ({len(content)/1024:.0f} KB)')

os.chdir(EXTRACT_DIR)
print(f'工作目录: {os.getcwd()}')

In [ ]:
# 检查 NumPy 版本，必要时重启
import numpy as np
if np.__version__ != '1.26.4':
    print(f'NumPy 版本不匹配 ({np.__version__})，请重启运行时后重新运行')
    os.kill(os.getpid(), 9)
print(f'NumPy: {np.__version__} ✓')

# 重启后再次设置工作目录
os.chdir(EXTRACT_DIR)
print(f'工作目录: {os.getcwd()}')

# 导入 MDLM 模块
import torch
import hydra
import lightning as L
from omegaconf import DictConfig

import dataloader
import diffusion
import main
import utils

print('所有模块导入成功 ✓')

---
## 辅助函数

In [ ]:
import time
def make_config(overrides):
    cfg_path = os.path.join(os.getcwd(), 'configs')
    print(f'配置路径: {cfg_path}')
    with hydra.initialize_config_dir(config_dir=cfg_path, version_base=None):
        return hydra.compose(config_name='config', overrides=overrides)
def run_experiment(config, tag=''):
    """运行一次生成实验，返回耗时和文本。"""
    L.seed_everything(config.seed, workers=True)
    tokenizer = dataloader.get_tokenizer(config)
    logger = utils.get_logger(tag or 'experiment')

    # 加载模型
    model = diffusion.Diffusion(config, tokenizer=tokenizer).to('cuda')

    # 采样
    t0 = time.time()
    if config.sampling.visualize:
        samples, snapshots = model.restore_model_and_sample(
            num_steps=config.sampling.steps, ret_snapshots=True)
        model.display_snapshots(snapshots, max_length=80)
        model.save_snapshots_to_file(snapshots,
            f'snapshot_{tag}.txt', max_length=0)
        print(f'快照已保存到 snapshot_{tag}.txt')
    else:
        samples = model.restore_model_and_sample(
            num_steps=config.sampling.steps)
    elapsed = time.time() - t0

    text = model.tokenizer.batch_decode(samples)
    model.compute_generative_perplexity(text)
    ppl = model.gen_ppl_metric.compute().item()
    print(f'耗时: {elapsed:.1f}s | PPL: {ppl:.1f}')
    print(f'文本: {text[0][:200]}...')
    return elapsed, ppl, text, snapshots if config.sampling.visualize else None

---
## 实验 1a：多 Seed 可视化对比

同一设置下不同随机种子生成不同文本，展示扩散模型的非自回归随机性。

In [ ]:
# 这大概需要 1 分钟
base_overrides = [
    'mode=sample_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-owt',
    'data=openwebtext-split',
    'model.length=256',
    'sampling.predictor=ddpm_cache',
    'sampling.steps=1000',
    'loader.eval_batch_size=1',
    'sampling.num_sample_batches=1',
    'backbone=hf_dit',
    'sampling.visualize=True',
]

results_seeds = {}
for seed in [42, 123, 999]:
    print(f'\n{"="*60}')
    print(f'Seed = {seed}')
    print(f'{"="*60}')
    cfg = make_config(base_overrides + [f'seed={seed}'])
    elapsed, ppl, text, _ = run_experiment(cfg, tag=f'seed{seed}')
    results_seeds[seed] = {'time': elapsed, 'ppl': ppl, 'text': text[0]}

print('\n' + '=' * 60)
print('实验 1a 总结：多 Seed 对比')
print('=' * 60)
for seed, r in results_seeds.items():
    print(f'Seed {seed:3d} | 耗时 {r["time"]:.1f}s | PPL {r["ppl"]:.0f}')
    print(f'        文本: {r["text"][:120]}...')

---
## 实验 1b：采样步数对比

固定 seed=42，对比 128 / 500 / 1000 步的生成质量差异。

In [ ]:
# 这大概需要 1 分钟
results_steps = {}
for steps in [128, 500, 1000]:
    print(f'\n{"="*60}')
    print(f'Steps = {steps}')
    print(f'{"="*60}')
    cfg = make_config(base_overrides + [
        f'seed=42',
        f'sampling.steps={steps}',
    ])
    elapsed, ppl, text, _ = run_experiment(cfg, tag=f'steps{steps}')
    results_steps[steps] = {'time': elapsed, 'ppl': ppl, 'text': text[0]}

print('\n' + '=' * 60)
print('实验 1b 总结：步数对比 (seed=42)')
print('=' * 60)
print(f'{"步数":>6s} | {"耗时":>8s} | {"PPL":>8s} | 文本预览')
print(f'{"-"*6}-+-{"-"*8}-+-{"-"*8}-+-{"-"*30}')
for steps, r in sorted(results_steps.items()):
    print(f'{steps:6d} | {r["time"]:7.1f}s | {r["ppl"]:8.0f} | {r["text"][:80]}...')

---
## 实验 2：采样器效率对比

对比 ddpm_cache / ddpm / analytic 三种采样器的速度和 PPL。

**原理说明**:
- `ddpm_cache`: 缓存模型输出 p(x0|xt)，token 不变时跳过前向计算 → 最快
- `ddpm`: 标准 ancestral 采样，每步都跑网络 → 中等
- `analytic`: SEDD 式解析采样，使用得分函数 → 最慢

理论上同 seed 下三者应生成相同文本（等价性已验证）。

In [ ]:
# 这大概需要 1 分钟
sampler_overrides = [
    'mode=sample_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-owt',
    'data=openwebtext-split',
    'model.length=256',
    'sampling.steps=1000',
    'loader.eval_batch_size=1',
    'sampling.num_sample_batches=1',
    'backbone=hf_dit',
    'seed=42',
    'sampling.visualize=False',
]

results_samplers = {}
for predictor in ['ddpm_cache', 'ddpm', 'analytic']:
    print(f'\n{"="*60}')
    print(f'Sampler = {predictor}')
    print(f'{"="*60}')
    cfg = make_config(sampler_overrides + [
        f'sampling.predictor={predictor}',
    ])
    elapsed, ppl, text, _ = run_experiment(cfg, tag=predictor)
    results_samplers[predictor] = {'time': elapsed, 'ppl': ppl, 'text': text[0]}

print('\n' + '=' * 60)
print('实验 2 总结：采样器对比 (seed=42, steps=1000)')
print('=' * 60)
print(f'{"采样器":>12s} | {"耗时":>8s} | {"PPL":>8s} | 相对速度')
print(f'{"-"*12}-+-{"-"*8}-+-{"-"*8}-+-{"-"*10}')
fastest = min(r['time'] for r in results_samplers.values())
for name, r in results_samplers.items():
    speedup = r['time'] / fastest
    print(f'{name:>12s} | {r["time"]:7.1f}s | {r["ppl"]:8.0f} | {speedup:.2f}x')

# 验证三者生成文本是否一致
texts = [r['text'] for r in results_samplers.values()]
if len(set(texts)) == 1:
    print('\n✅ 三者生成文本完全相同（等价性验证通过）')
else:
    print('\n⚠️ 生成文本有差异（analytic 可能因浮点误差略有不同）')

---
## 拓展实验：生成长文本 (Length=1024)

增加序列长度可以提升生成文本的连贯性。

In [ ]:
# 这大概需要 1 分钟
cfg_long = make_config([
    'mode=sample_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-owt',
    'data=openwebtext-split',
    'model.length=1024',
    'sampling.predictor=ddpm_cache',
    'sampling.steps=1000',
    'loader.eval_batch_size=1',
    'sampling.num_sample_batches=1',
    'backbone=hf_dit',
    'seed=42',
    'sampling.visualize=True',
])

print('生成长文本 (length=1024)，这可能需要 1-2 分钟...')
elapsed, ppl, text, _ = run_experiment(cfg_long, tag='length1024')
print(f'\n耗时: {elapsed:.1f}s | PPL: {ppl:.0f}')
print(f'\n完整文本:\n{text[0]}')

---
## 结果汇总

所有快照文件已保存到当前目录：
- `snapshot_seed42.txt`, `snapshot_seed123.txt`, `snapshot_seed999.txt`
- `snapshot_steps128.txt`, `snapshot_steps500.txt`, `snapshot_steps1000.txt`
- `snapshot_length1024.txt`

下载这些文件即可用于课程报告插图。

In [ ]:
import os, zipfile
snapshots = [f for f in os.listdir('.') if f.startswith('snapshot_') and f.endswith('.txt')]
print(f'找到 {len(snapshots)} 个快照文件:')
for s in snapshots:
    print(f'  {s} ({os.path.getsize(s)} bytes)')
with zipfile.ZipFile('mdlm_snapshots.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for s in snapshots:
        zf.write(s)
size_kb = os.path.getsize('mdlm_snapshots.zip') / 1024
print(f'\n已打包: mdlm_snapshots.zip ({size_kb:.1f} KB)')
# 触发 Colab 下载
from google.colab import files
files.download('mdlm_snapshots.zip')

---
## 实验 3：OWT Perplexity 基准对比 (Table 2)

在 OpenWebText 验证集上计算 MDLM 的 ELBO perplexity，与论文报告的 AR 和 SEDD 基线对比。

**论文 Table 2 参考值**:
| 模型 | OWT PPL (↓) |
|:----|:-----------:|
| AR (Transformer) | 17.54 |
| SEDD | ≤24.10 |
| **MDLM (Ours)** | **≤23.21** |

> ⚠️ 这会遍历整个 OWT 验证集 (100K 文档)，T4 上可能需要 1-2 小时。
> 你可以设置 `limit_val_batches=0.1` 只跑 10% 来快速验证。


In [ ]:
# OWT Perplexity 评估 — ppl_eval 模式
# 这会在 OWT 验证集上计算 MDLM 的 ELBO (变分下界)，输出 val/ppl
# ⏱ T4 完整验证集约 1-2 小时，10% 约 6-12 分钟

owt_overrides = [
    'mode=ppl_eval',
    'eval.checkpoint_path=kuleshov-group/mdlm-owt',
    'data=openwebtext-split',
    'model=small',
    'parameterization=subs',
    'backbone=hf_dit',
    'model.length=1024',
    'loader.eval_batch_size=4',  # T4 显存限制
    'trainer.limit_val_batches=0.1',  # 快速验证：只跑 10% 数据
    # 去掉下面行的注释可跑完整验证集
    # 'trainer.limit_val_batches=1.0',
    '+wandb.offline=true',
]

cfg = make_config(owt_overrides)
tokenizer = dataloader.get_tokenizer(cfg)

print('正在评估 MDLM 在 OWT 验证集上的 Perplexity...')
print('使用 10% 验证集（快速模式）')
print('=' * 60)
t0 = time.time()

# 使用 main._ppl_eval 逻辑
model = diffusion.Diffusion(cfg, tokenizer=tokenizer).to('cuda')
if cfg.eval.disable_ema:
    model.ema = None

# Trainer
trainer = hydra.utils.instantiate(
    cfg.trainer,
    default_root_dir=os.getcwd(),
    callbacks=[],
    strategy=hydra.utils.instantiate(cfg.strategy),
    logger=False)

# 加载数据（只验证集）
_, valid_ds = dataloader.get_dataloaders(
    cfg, tokenizer, skip_train=True, valid_seed=cfg.seed)

# 评估
metrics = trainer.validate(model, valid_ds)
elapsed = time.time() - t0

print(f'\n{"=" * 60}')
print(f'OWT Perplexity 评估完成！耗时: {elapsed/60:.1f} 分钟')
print(f'{"=" * 60}')
print(f'MDLM OWT PPL: {metrics[0]["val/ppl"]:.2f}')
print(f'MDLM OWT NLL: {metrics[0]["val/nll"]:.4f}')
print()
print('Table 2 对比:')
print('  AR (论文值):       17.54')
print('  SEDD (论文值):     ≤24.10')
print(f'  MDLM (你的结果):   ≤{metrics[0]["val/ppl"]:.2f}')


---
## 实验 4：Zero-shot Perplexity 跨数据集评估 (Table 3)

用 OWT 上训练的 MDLM 模型，在 7 个未见过的数据集上评估泛化能力。

**论文 Table 3 参考值**:

| 模型 | PTB | Wikitext | LM1B | Lambada | AG News | Pubmed | Arxiv |
|:----|:---:|:--------:|:----:|:-------:|:-------:|:------:|:-----:|
| AR | 82.05 | 25.75 | 51.25 | 51.28 | 52.09 | 49.01 | 41.73 |
| SEDD | 100.09 | 34.28 | 68.20 | 49.86 | 62.09 | 44.53 | 38.48 |
| **MDLM** | **95.26** | **32.83** | **67.01** | **47.52** | **61.15** | **41.89** | **37.37** |

> 每个数据集评估约需 3-10 分钟，7 个总共约 30-60 分钟。


In [ ]:
# Zero-shot Perplexity 评估 — 7 个数据集
# ⏱ 每个约 3-10 分钟，总共约 30-60 分钟

ZERO_SHOT_DATASETS = [
    ('ptb', 'Penn Treebank'),
    ('wikitext2', 'Wikitext-2'),
    ('lm1b-gpt2', 'LM1B (GPT2 tok)'),
    ('lambada', 'Lambada'),
    ('ag_news', 'AG News'),
    ('scientific_papers_pubmed', 'PubMed'),
    ('scientific_papers_arxiv', 'Arxiv'),
]

# 论文参考值 (Table 3)
paper_ar = {
    'ptb': 82.05, 'wikitext2': 25.75, 'lm1b-gpt2': 51.25,
    'lambada': 51.28, 'ag_news': 52.09,
    'scientific_papers_pubmed': 49.01, 'scientific_papers_arxiv': 41.73
}
paper_sedd = {
    'ptb': 100.09, 'wikitext2': 34.28, 'lm1b-gpt2': 68.20,
    'lambada': 49.86, 'ag_news': 62.09,
    'scientific_papers_pubmed': 44.53, 'scientific_papers_arxiv': 38.48
}

results_zeroshot = {}

for ds_name, ds_label in ZERO_SHOT_DATASETS:
    print(f'\n{"=" * 60}')
    print(f'评估: {ds_label} ({ds_name})')
    print(f'{"=" * 60}')

    cfg = make_config([
        'mode=ppl_eval',
        'eval.checkpoint_path=kuleshov-group/mdlm-owt',
        f'data={ds_name}',
        'model=small',
        'parameterization=subs',
        'backbone=hf_dit',
        'model.length=1024',
        'loader.eval_batch_size=4',
        '+wandb.offline=true',
    ])
    tokenizer = dataloader.get_tokenizer(cfg)

    try:
        model = diffusion.Diffusion(cfg, tokenizer=tokenizer).to('cuda')
        if cfg.eval.disable_ema:
            model.ema = None

        trainer = hydra.utils.instantiate(
            cfg.trainer,
            default_root_dir=os.getcwd(),
            callbacks=[],
            strategy=hydra.utils.instantiate(cfg.strategy),
            logger=False)

        _, valid_ds = dataloader.get_dataloaders(
            cfg, tokenizer, skip_train=True, valid_seed=cfg.seed)

        t0 = time.time()
        metrics = trainer.validate(model, valid_ds)
        elapsed = time.time() - t0

        ppl = metrics[0]['val/ppl']
        results_zeroshot[ds_name] = ppl
        print(f'✓ {ds_label}: PPL = {ppl:.2f} (耗时 {elapsed/60:.1f} 分)')
    except Exception as e:
        print(f'✗ {ds_label} 评估失败: {e}')
        results_zeroshot[ds_name] = None

# 汇总结果
print('\n' + '=' * 60)
print('实验 4 总结：Zero-shot Perplexity (Table 3)')
print('=' * 60)
print(f'{"数据集":>20s} | {"MDLM(你的)":>10s} | {"AR(论文)":>10s} | {"SEDD(论文)":>10s}')
print('-' * 60)
for ds_name, ds_label in ZERO_SHOT_DATASETS:
    mine = results_zeroshot.get(ds_name, None)
    mine_str = f'{mine:.2f}' if mine else '  N/A'
    ar_str = f'{paper_ar[ds_name]:.2f}'
    sedd_str = f'{paper_sedd[ds_name]:.2f}'
    print(f'{ds_label:>20s} | {mine_str:>10s} | {ar_str:>10s} | {sedd_str:>10s}')

paper_mdlm = {
    'ptb': 95.26, 'wikitext2': 32.83, 'lm1b-gpt2': 67.01,
    'lambada': 47.52, 'ag_news': 61.15,
    'scientific_papers_pubmed': 41.89, 'scientific_papers_arxiv': 37.37
}
print()
print('与论文 MDLM 报告值对比:')
for ds_name, ds_label in ZERO_SHOT_DATASETS:
    mine = results_zeroshot.get(ds_name, None)
    expected = paper_mdlm[ds_name]
    if mine:
        diff = mine - expected
        marker = '✓' if abs(diff) < 1.0 else '⚠️'
        print(f'  {ds_label:>20s}: 你的={mine:.2f} vs 论文={expected:.2f} (差={diff:+.2f}) {marker}')
    else:
        print(f'  {ds_label:>20s}: N/A (评估失败)')


In [ ]:
# 下载 AR 和 SEDD 基线预训练权重（来自论文作者 Google Drive）
# 文件: ar.ckpt (~440MB), sedd.ckpt (~440MB)
# ⏱ 下载约需 5-15 分钟，取决于网络
import os, subprocess

GDRIVE_FOLDER = "https://drive.google.com/drive/folders/16LuuptK7Xfk-vzhQYZBZ0SA-B-BFluau"
CKPT_DIR = "baseline_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(f"{CKPT_DIR}/ar.ckpt") or not os.path.exists(f"{CKPT_DIR}/sedd.ckpt"):
    print("正在从 Google Drive 下载基线权重...")
    subprocess.run([
        "gdown", "--folder", GDRIVE_FOLDER,
        "-O", CKPT_DIR, "--remaining-ok"
    ], check=True)
    print("下载完成 ✅")
else:
    print("基线权重已存在 ✅")

!ls -lh "{CKPT_DIR}"


In [ ]:
# AR 基线评估 — OWT 验证集 Perplexity
# 需要先运行上一格下载 ar.ckpt
# ⏱ T4 上跑 10% 验证集约 6-12 分钟

ar_cfg = make_config([
    'mode=ppl_eval',
    f'eval.checkpoint_path={CKPT_DIR}/ar.ckpt',
    'data=openwebtext-split',
    'model=small-ar',
    'parameterization=ar',
    'backbone=ar',
    'model.length=1024',
    'loader.eval_batch_size=4',
    'trainer.limit_val_batches=0.1',  # 快速验证
    '+wandb.offline=true',
])

tokenizer_ar = dataloader.get_tokenizer(ar_cfg)
print('正在评估 AR 基线...')
t0 = time.time()

# 本地 checkpoint：用 main._load_from_checkpoint 加载
model_ar = main._load_from_checkpoint(ar_cfg, tokenizer_ar)
if ar_cfg.eval.disable_ema:
    model_ar.ema = None

trainer_ar = hydra.utils.instantiate(
    ar_cfg.trainer,
    default_root_dir=os.getcwd(),
    callbacks=[],
    strategy=hydra.utils.instantiate(ar_cfg.strategy),
    logger=False)

_, valid_ds_ar = dataloader.get_dataloaders(
    ar_cfg, tokenizer_ar, skip_train=True, valid_seed=ar_cfg.seed)

metrics_ar = trainer_ar.validate(model_ar, valid_ds_ar)
elapsed_ar = time.time() - t0
ppl_ar = metrics_ar[0]['val/ppl']

print(f'\n{"=" * 60}')
print(f'AR 基线评估完成！耗时: {elapsed_ar/60:.1f} 分钟')
print(f'AR OWT PPL: {ppl_ar:.2f}')
print(f'论文参考值: 17.54')


In [ ]:
# SEDD 基线评估 — OWT 验证集 Perplexity
# 需要先下载 sedd.ckpt
# ⏱ T4 上跑 10% 验证集约 6-12 分钟

sedd_cfg = make_config([
    'mode=ppl_eval',
    f'eval.checkpoint_path={CKPT_DIR}/sedd.ckpt',
    'data=openwebtext-split',
    'model=small',
    'parameterization=sedd',
    'backbone=dit',
    'model.length=1024',
    'loader.eval_batch_size=4',
    'time_conditioning=True',
    'trainer.limit_val_batches=0.1',
    '+wandb.offline=true',
])

tokenizer_sedd = dataloader.get_tokenizer(sedd_cfg)
print('正在评估 SEDD 基线...')
t0 = time.time()

model_sedd = main._load_from_checkpoint(sedd_cfg, tokenizer_sedd)
if sedd_cfg.eval.disable_ema:
    model_sedd.ema = None

trainer_sedd = hydra.utils.instantiate(
    sedd_cfg.trainer,
    default_root_dir=os.getcwd(),
    callbacks=[],
    strategy=hydra.utils.instantiate(sedd_cfg.strategy),
    logger=False)

_, valid_ds_sedd = dataloader.get_dataloaders(
    sedd_cfg, tokenizer_sedd, skip_train=True, valid_seed=sedd_cfg.seed)

metrics_sedd = trainer_sedd.validate(model_sedd, valid_ds_sedd)
elapsed_sedd = time.time() - t0
ppl_sedd = metrics_sedd[0]['val/ppl']

print(f'\n{"=" * 60}')
print(f'SEDD 基线评估完成！耗时: {elapsed_sedd/60:.1f} 分钟')
print(f'SEDD OWT PPL: {ppl_sedd:.2f}')
print(f'论文参考值: ≤24.10')


---
## Table 2 结果汇总

将实验 3 和 AR/SEDD 的结果填入下表：

| 模型 | OWT PPL (↓) vs 论文 |
|:----|:-------------------:|
| **AR (Transformer)** | `你的值` vs 17.54 |
| **SEDD** | `你的值` vs ≤24.10 |
| **MDLM (Ours)** | `你的值` vs **≤23.21** |

以及 Table 3 Zero-shot 结果。

### Table 3 参考值

| 模型 | PTB | Wikitext | LM1B | Lambada | AG News | Pubmed | Arxiv |
|:----|:---:|:--------:|:----:|:-------:|:-------:|:------:|:-----:|
| AR | 82.05 | 25.75 | 51.25 | 51.28 | 52.09 | 49.01 | 41.73 |
| SEDD | 100.09 | 34.28 | 68.20 | 49.86 | 62.09 | 44.53 | 38.48 |
| **MDLM** | **95.26** | **32.83** | **67.01** | **47.52** | **61.15** | **41.89** | **37.37** |
